# RESP電荷のための波動関数

`dft_sandbox.ipynb` で B3LYP/6-31G(d) により最適化した座標を読み、構造はそのままに **Hartree–Fock/6-31G(d)** の単点計算を行う。出力する fchk を Multiwfn に渡し、RESP電荷をフィットする。

AMBER でよく使う RESP は、気相の HF/6-31G(d) の静電ポテンシャルに合わせた原子電荷である。Gaussian の `pop=mk iop(6/33=2,6/42=6)` は、その静電ポテンシャル格子をログへ書き出す指定にすぎない。Multiwfn は fchk の軌道から格子を自分で作るので、この指定は要らない。

座標の既定値は MACE で最適化済みの `output_mace/TAPC/TAPC_opt.xyz`。別の材料にするときは、このパスだけ変える。

出力は座標と同じフォルダに置く。

- `{名前}_resp.log` : SCFのログ
- `{名前}_resp.chk` : PySCFのチェックポイント
- `{名前}_resp.fchk` : Multiwfn に渡す波動関数

In [1]:
import os
import gc
import numpy as np
import cupy

# 座標と基底関数から分子を作る
from pyscf import gto
# Hartree-Fock。RESPの標準的な波動関数はDFTではなくこちら
from gpu4pyscf.scf import RHF
# XC_FUNCTIONAL を b3lyp に戻したときだけ使う
from gpu4pyscf.dft import rks

In [2]:
def release_gpu():
    """計算オブジェクトを捨てたあと、CuPyが確保したままのVRAMを返す。"""
    gc.collect()
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()

def write_fchk(mf, filename, title="PySCF SCF"):
    """収束したSCFからGaussian形式のfchkを書く。

    Multiwfnは基底関数の並びがGaussianと同じであることを前提に軌道を読む。
    PySCFのd/f関数の並びはそれと違うので、Molden出力と同じ順へ入れ替えてから書く。
    """
    from pyscf.tools.molden import order_ao_index

    def write_ints(handle, values):
        # fchkの整数配列は1行に6個
        for i, value in enumerate(values, 1):
            handle.write(f"{int(value):12d}")
            if i % 6 == 0:
                handle.write("\n")
        if len(values) % 6:
            handle.write("\n")

    def write_floats(handle, values):
        # fchkの実数配列は1行に5個
        flat = np.asarray(values, dtype=float).ravel()
        for i, value in enumerate(flat, 1):
            handle.write(f"{value:16.8E}")
            if i % 5 == 0:
                handle.write("\n")
        if len(flat) % 5:
            handle.write("\n")

    def section_array(handle, name, kind, values):
        values = list(values)
        handle.write(f"{name:<43}{kind}   N={len(values):12d}\n")
        (write_ints if kind == "I" else write_floats)(handle, values)

    def section_scalar(handle, name, kind, value):
        if kind == "I":
            handle.write(f"{name:<43}I {int(value):17d}\n")
        else:
            handle.write(f"{name:<43}R {float(value):22.15E}\n")

    mol = mf.mol
    # GPU上の軌道をCPUへコピーしてからファイルへ書く
    mo_coeff = mf.mo_coeff.get() if hasattr(mf.mo_coeff, "get") else np.asarray(mf.mo_coeff)
    mo_energy = mf.mo_energy.get() if hasattr(mf.mo_energy, "get") else np.asarray(mf.mo_energy)
    mo_coeff = mo_coeff[order_ao_index(mol)]
    nbf, nmo = mo_coeff.shape
    n_alpha, n_beta = mol.nelec

    # 各縮約殻の角運動量、原始関数、原子への対応。fchkはこれを軌道係数と一緒に持つ
    shell_type, prim_per, shell_atom = [], [], []
    exps, coeffs, shell_xyz = [], [], []
    nbf_check = 0
    for ib in range(mol.nbas):
        l = mol.bas_angular(ib)
        nctr = mol.bas_nctr(ib)
        es = mol.bas_exp(ib)
        cs = mol.bas_ctr_coeff(ib)
        xyz = mol.atom_coord(mol.bas_atom(ib))
        nfunc = (l + 1) * (l + 2) // 2 if mol.cart else 2 * l + 1
        for ic in range(nctr):
            keep = np.flatnonzero(np.abs(cs[:, ic]) > 0)
            if keep.size == 0:
                continue
            # 球面調和のd/fは負の角運動量で表す。sとpはそのまま
            shell_type.append(l if mol.cart or l <= 1 else -l)
            prim_per.append(int(keep.size))
            shell_atom.append(mol.bas_atom(ib) + 1)
            exps.extend(es[keep])
            coeffs.extend(cs[keep, ic])
            shell_xyz.extend(xyz)
            nbf_check += nfunc
    if nbf_check != nbf:
        raise RuntimeError(f"基底関数の数が一致しません: shells={nbf_check}, nao={nbf}")

    xc = getattr(mf, "xc", "HF")
    basis = mol.basis if isinstance(mol.basis, str) else "Gen"
    with open(filename, "w") as handle:
        handle.write(title + "\n")
        handle.write(f"SCF       R{str(xc).upper():<16}{basis}\n")
        section_scalar(handle, "Number of atoms", "I", mol.natm)
        section_scalar(handle, "Charge", "I", mol.charge)
        section_scalar(handle, "Multiplicity", "I", mol.spin + 1)
        section_scalar(handle, "Number of electrons", "I", mol.nelectron)
        section_scalar(handle, "Number of alpha electrons", "I", n_alpha)
        section_scalar(handle, "Number of beta electrons", "I", n_beta)
        section_scalar(handle, "Number of basis functions", "I", nbf)
        section_scalar(handle, "Number of independent functions", "I", nmo)
        section_array(handle, "Atomic numbers", "I", [mol.atom_charge(i) for i in range(mol.natm)])
        section_array(handle, "Nuclear charges", "R", mol.atom_charges())
        section_array(handle, "Current cartesian coordinates", "R", mol.atom_coords().ravel())
        section_scalar(handle, "Number of contracted shells", "I", len(shell_type))
        section_scalar(handle, "Number of primitive shells", "I", len(exps))
        section_array(handle, "Shell types", "I", shell_type)
        section_array(handle, "Number of primitives per shell", "I", prim_per)
        section_array(handle, "Shell to atom map", "I", shell_atom)
        section_array(handle, "Primitive exponents", "R", exps)
        section_array(handle, "Contraction coefficients", "R", coeffs)
        section_array(handle, "Coordinates of each shell", "R", shell_xyz)
        section_array(handle, "Alpha Orbital Energies", "R", mo_energy)
        # 列優先。1本の軌道の係数が並んでから、次の軌道に進む
        section_array(handle, "Alpha MO coefficients", "R", mo_coeff.ravel(order="F"))
        section_scalar(handle, "Total Energy", "R", mf.e_tot)

In [3]:
# 計算したい材料の最適化された座標を選ぶ
xyz_path = "output_mace/TAPC/TAPC_opt.xyz"

# RESPの標準は Hartree-Fock。B3LYPに戻す場合は "b3lyp" にする
XC_FUNCTIONAL = "hf"
# 6-31+G(d)。分極関数と分散関数をつける
BASIS_SET = "6-31+g(d,p)"
# xyzには電荷とスピンが入っていない。中性の閉殻なら 0 と 0
CHARGE = 0
SPIN = 0

# NPD_opt.xyz なら材料名は NPD。ログとfchkは座標と同じフォルダへ置く
stem = os.path.splitext(os.path.basename(xyz_path))[0]
mol_name = stem[:-4] if stem.endswith("_opt") else stem
out_dir = os.path.dirname(xyz_path)
os.makedirs(out_dir, exist_ok=True)
print(f"材料: {mol_name}")
print(f"座標: {xyz_path}")
print(f"計算レベル: {XC_FUNCTIONAL}/{BASIS_SET}  charge={CHARGE} spin={SPIN}")

材料: TAPC
座標: output_mace/TAPC/TAPC_opt.xyz
計算レベル: hf/6-31+g(d,p)  charge=0 spin=0


In [4]:
# 最適化済みの座標をそのまま使う。このノートブックでは構造最適化をしない
print(f"単点計算を開始: {XC_FUNCTIONAL}/{BASIS_SET}")
log_path = os.path.join(out_dir, f"{mol_name}_resp.log")
mol = gto.M(
    atom=xyz_path,
    basis=BASIS_SET,
    charge=CHARGE,
    spin=SPIN,
    verbose=4,
    output=log_path,
)

# Hartree-Fockには交換相関の数値積分格子がない。DFTに戻したときだけ格子を置く
if XC_FUNCTIONAL.lower() in ("hf", "rhf"):
    mf = RHF(mol)
else:
    mf = rks.RKS(mol, xc=XC_FUNCTIONAL)
    mf.grids.level = 3

# 密度フィッティングは使わない。補助基底の3中心積分は8GBのGPUに載らない
mf.chkfile = os.path.join(out_dir, f"{mol_name}_resp.chk")
mf.kernel()

fchk_path = os.path.join(out_dir, f"{mol_name}_resp.fchk")
write_fchk(mf, fchk_path, title=f"{mol_name} {XC_FUNCTIONAL}/{BASIS_SET}")
print(f"E = {float(mf.e_tot):.8f} Hartree")
print(f"fchk: {fchk_path}")
print("このfchkをMultiwfnで開き、RESP電荷を計算する。")

del mf, mol
release_gpu()

単点計算を開始: hf/6-31+g(d,p)
overwrite output file: output_mace/TAPC/TAPC_resp.log


WARN: 12 small eigenvectors of overlap matrix removed because of linear dependency between AOs.
The support for low-rank overlap matrix is not fully tested. Please report any bug you encountered to the developers.


E = -1877.73812464 Hartree
fchk: output_mace/TAPC/TAPC_resp.fchk
このfchkをMultiwfnで開き、RESP電荷を計算する。
